
# Egocentric STA on Colab — Tracks A/B/C (YOLO ➜ Head ➜ Pruning)

This notebook follows your plan and builds a runnable pipeline for **STA** with:

- **Track A:** YOLO detector (last frame) ➜ lightweight head (next-active + verb + TTC)
- **Track B:** Fusion hooks placeholders (enable later)
- **Track C:** RGTP-style token pruning demo

**Two modes**
- **DEMO=True:** runs end-to-end with synthetic labels if real ones are missing.
- **DEMO=False:** expects real data under `/content/drive/MyDrive/ego4d_data`.

> Start DEMO first, then flip to FULL when your labels/manifests are ready.

Next blocks sets up SSH to GitHub (**Deploy Key stored in Drive**), pulls your repo, runs code, logs outputs, and pushes logs/artifacts back to GitHub.
- One-time: add your **Deploy Key** public key to your GitHub repo Deploy Keys with **write access**.
- Edit the variables (**GITHUB_USER**, **REPO_NAME**) below once; they persist per notebook.


In [1]:
# ⛳ Mount Drive and set variables
from google.colab import drive
drive.mount('/content/drive')

GITHUB_USER = "saeedzns"          # <<< EDIT ME
REPO_NAME   = "Ego4d-LiteSTA"             # <<< default as requested
REPO_SSH    = f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
WORKDIR     = f"/content/drive/MyDrive/{REPO_NAME}"
SSH_DIR     = "/content/drive/MyDrive/.ssh"

print("Repo:", REPO_SSH)
print("Workdir:", WORKDIR)


# 🔐 SSH + git config + clone/pull
# 1) Git identity (optional but recommended)
!git config --global user.name "saeedzns"
!git config --global user.email "zns1992@gmail.com"

Mounted at /content/drive
Repo: git@github.com:saeedzns/Ego4d-LiteSTA.git
Workdir: /content/drive/MyDrive/Ego4d-LiteSTA


In [2]:
# 2) SSH agent and key from Drive (key filename: colab_egolite)
%%bash -s "$SSH_DIR"
SSH_DIR="$1"
mkdir -p "$SSH_DIR"
chmod 700 "$SSH_DIR"
eval "$(ssh-agent -s)"
ssh-add "$SSH_DIR/colab_egolite" >/dev/null 2>&1 || true
mkdir -p ~/.ssh && chmod 700 ~/.ssh
cat <<EOF > ~/.ssh/config
Host github.com
  HostName github.com
  User git
  IdentityFile $SSH_DIR/colab_egolite
  IdentitiesOnly yes
StrictHostKeyChecking accept-new
EOF
chmod 600 ~/.ssh/config
ssh-keyscan -t rsa,ed25519 github.com >> ~/.ssh/known_hosts 2>/dev/null || true
chmod 644 ~/.ssh/known_hosts

Agent pid 1104


## Pull and Push — Daily Sync
Run after setting variables and SSH above.


In [ ]:
%cd "$WORKDIR"
!bash pull.sh


/content/drive/MyDrive/Ego4d-LiteSTA
[pull] Fetching and pulling latest for 'main'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 3 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 4.26 KiB | 75.00 KiB/s, done.
From github.com:saeedzns/Ego4d-LiteSTA
   89a7679..fee57f7  main       -> origin/main
From github.com:saeedzns/Ego4d-LiteSTA
 * branch            main       -> FETCH_HEAD
Updating 89a7679..fee57f7
Created autostash: e5f2d3b
Fast-forward
 Glossary.md | 162 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 162 insertions(+)
 create mode 100644 Glossary.md
Applying autostash resulted in conflicts.
Your changes are safe in the stash.
You can run "git stash pop" or "git stash drop" at any time.
[pull] Done.


In [3]:
%cd "$WORKDIR"
!bash push.sh


/content/drive/MyDrive/Ego4d-LiteSTA
[push] Staging all changes...
[push] Committing with message: chore: daily sync 2025-11-02 20:06:06Z
[main 92629c4] chore: daily sync 2025-11-02 20:06:06Z
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite notebooks/ego4d_sta_colab.ipynb (100%)
[push] Updating branch with latest remote (rebase)...
From github.com:saeedzns/Ego4d-LiteSTA
 * branch            main       -> FETCH_HEAD
Current branch main is up to date.
[push] Pushing to origin/main...
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 4.86 KiB | 237.00 KiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To github.com:saeedzns/Ego4d-LiteSTA.git
   fee57f7..92629c4  main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.
[push] Done.


## 0. Config & Paths

In [ ]:
DEMO = True  #@param {type:"boolean"}
EGO4D_ROOT = "/content/drive/MyDrive/ego4d_data"  #@param {type:"string"}
PROJ_ROOT = "/content/drive/MyDrive/thesis_sta"   #@param {type:"string"}

import os
DATA_OUT = f"{PROJ_ROOT}/data"
RUNS_DIR = f"{PROJ_ROOT}/runs"
CKPT_DIR = f"{PROJ_ROOT}/checkpoints"
LOGS_DIR = f"{PROJ_ROOT}/logs"
MANI_DIR = f"{PROJ_ROOT}/manifests"
CONFIG_DIR = f"{PROJ_ROOT}/configs"
for p in [PROJ_ROOT, DATA_OUT, RUNS_DIR, CKPT_DIR, LOGS_DIR, MANI_DIR, CONFIG_DIR]:
    os.makedirs(p, exist_ok=True)

print("DEMO:", DEMO)
print("EGO4D_ROOT:", EGO4D_ROOT)
print("PROJ_ROOT:", PROJ_ROOT)

## 1. System & Dependencies

In [ ]:
!nvidia-smi || true
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

In [ ]:
# Install core deps
!pip -q install -U ultralytics==8.3.0 timm==1.0.11 einops yacs pyyaml tqdm opencv-python pycocotools matplotlib onnxruntime-gpu --progress-bar off
import torch, torchvision, timm, ultralytics
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("ultralytics:", ultralytics.__version__)

## 3. Data Discovery

In [ ]:
import os, glob, random, shutil
from pathlib import Path

frames_dir_candidates = [
    f"{EGO4D_ROOT}/frames",
    f"{EGO4D_ROOT}/images",
    f"{EGO4D_ROOT}/sta/frames",
]
label_dirs = [
    f"{EGO4D_ROOT}/labels",
    f"{EGO4D_ROOT}/yolo_labels",
]
ann_files = [
    f"{EGO4D_ROOT}/sta_v2.json",
    f"{EGO4D_ROOT}/annotations/sta_v2.json",
    f"{EGO4D_ROOT}/sta/sta_v2.json",
]

def first_existing(paths):
    for p in paths:
        if os.path.isdir(p) or os.path.isfile(p):
            return p
    return None

FRAMES_DIR = first_existing(frames_dir_candidates)
YOLO_LABEL_DIR = first_existing(label_dirs)
ANN_JSON = first_existing(ann_files)

print("FRAMES_DIR:", FRAMES_DIR)
print("YOLO_LABEL_DIR:", YOLO_LABEL_DIR)
print("ANN_JSON:", ANN_JSON)

# DEMO fallback
import numpy as np, cv2
DEMO_IMG_DIR = f"{DATA_OUT}/demo/images"
DEMO_LBL_DIR = f"{DATA_OUT}/demo/labels"
os.makedirs(DEMO_IMG_DIR, exist_ok=True)
os.makedirs(DEMO_LBL_DIR, exist_ok=True)

if DEMO or FRAMES_DIR is None:
    print("Preparing synthetic demo images...")
    for i in range(20):
        img = np.zeros((480, 640, 3), dtype=np.uint8)
        for _ in range(3):
            x1, y1 = np.random.randint(0, 540), np.random.randint(0, 380)
            x2, y2 = x1 + np.random.randint(40,100), y1 + np.random.randint(40,100)
            color = (np.random.randint(40,255), np.random.randint(40,255), np.random.randint(40,255))
            cv2.rectangle(img, (x1,y1), (x2,y2), color, -1)
        cv2.putText(img, f"demo_{i}", (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)
        cv2.imwrite(f"{DEMO_IMG_DIR}/{i:05d}.jpg", img)
        with open(f"{DEMO_LBL_DIR}/{i:05d}.txt", "w") as f:
            f.write("0 0.5 0.5 0.3 0.3\n")
    FRAMES_DIR = DEMO_IMG_DIR
    YOLO_LABEL_DIR = DEMO_LBL_DIR

images = sorted(glob.glob(os.path.join(FRAMES_DIR, "*.jpg")) + glob.glob(os.path.join(FRAMES_DIR, "*.png")))
random.seed(42); random.shuffle(images)
split = int(0.8*len(images))
train_imgs, val_imgs = images[:split], images[split:]
print(f"Found images: {len(images)} (train {len(train_imgs)} | val {len(val_imgs)})")

## 4. Prepare YOLO dataset YAML

In [ ]:
import yaml, os, shutil, glob

YOLO_DATA_YAML = f"{CONFIG_DIR}/yolo_sta.yaml"
YOLO_IMG_TRAIN = f"{DATA_OUT}/yolo/images/train"
YOLO_IMG_VAL   = f"{DATA_OUT}/yolo/images/val"
YOLO_LBL_TRAIN = f"{DATA_OUT}/yolo/labels/train"
YOLO_LBL_VAL   = f"{DATA_OUT}/yolo/labels/val"
for d in [YOLO_IMG_TRAIN, YOLO_IMG_VAL, YOLO_LBL_TRAIN, YOLO_LBL_VAL]:
    os.makedirs(d, exist_ok=True)

def copy_pair(img_path, dst_img_dir, dst_lbl_dir):
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl = os.path.join(YOLO_LABEL_DIR, base + ".txt") if YOLO_LABEL_DIR else None
    shutil.copy2(img_path, os.path.join(dst_img_dir, os.path.basename(img_path)))
    if lbl and os.path.exists(lbl):
        shutil.copy2(lbl, os.path.join(dst_lbl_dir, os.path.basename(lbl)))
    else:
        open(os.path.join(dst_lbl_dir, base + ".txt"), "w").close()

for p in train_imgs: copy_pair(p, YOLO_IMG_TRAIN, YOLO_LBL_TRAIN)
for p in val_imgs:   copy_pair(p, YOLO_IMG_VAL,   YOLO_LBL_VAL)

data_cfg = {
    "path": os.path.dirname(YOLO_DATA_YAML),
    "train": os.path.relpath(YOLO_IMG_TRAIN, start=os.path.dirname(YOLO_DATA_YAML)),
    "val":   os.path.relpath(YOLO_IMG_VAL,   start=os.path.dirname(YOLO_DATA_YAML)),
    "names": {0: "object"}
}
with open(YOLO_DATA_YAML, "w") as f: yaml.safe_dump(data_cfg, f)
print(open(YOLO_DATA_YAML).read())

## 5. Train YOLO (Track A - Stage A)

In [ ]:
from ultralytics import YOLO
IMGSZ=640
EPOCHS=5 if DEMO else 60
model = YOLO("yolov8s.pt")
res = model.train(
    data=YOLO_DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ,
    batch=8 if DEMO else 16, project=f"{RUNS_DIR}/yolo_sta",
    name="exp_sta", optimizer="adamw", cos_lr=True, patience=10, verbose=True
)
WEIGHTS = getattr(model, "ckpt_path", None)
print("Best weights:", WEIGHTS)

## 6. Export Top‑K candidates (val set)

In [ ]:
import json, numpy as np, glob
from ultralytics import YOLO
from tqdm import tqdm

TOPK=5
CAND_JSON = f"{MANI_DIR}/candidates_val_k{TOPK}.json"
weights_path = None
candidates = sorted(glob.glob(f"{RUNS_DIR}/yolo_sta/exp_sta/weights/best*.pt"))
if candidates:
    weights_path = candidates[-1]
else:
    # fall back
    weights_path = "yolov8s.pt"
print("Using weights:", weights_path)

det = YOLO(weights_path)
manifest=[]
subset = val_imgs[: (30 if DEMO else len(val_imgs))]
for p in tqdm(subset):
    out = det.predict(source=p, imgsz=IMGSZ, conf=0.01, iou=0.6, verbose=False)[0]
    if out.boxes is None:
        boxes = []; scores=[]; clss=[]
    else:
        boxes = out.boxes.xyxy.cpu().numpy()
        scores = out.boxes.conf.cpu().numpy()
        clss = out.boxes.cls.cpu().numpy()
    order = np.argsort(scores)[::-1][:TOPK] if len(boxes)>0 else []
    entry = {"image": p, "candidates": []}
    for i in order:
        entry["candidates"].append({
            "xyxy": boxes[i].tolist(),
            "score": float(scores[i]),
            "noun": int(clss[i])
        })
    manifest.append(entry)

with open(CAND_JSON,"w") as f: json.dump(manifest, f, indent=2)
print("Saved:", CAND_JSON, "#items:", len(manifest))

## 7. Utils: IoU + Top1 proxy

In [ ]:
import numpy as np

def box_iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1, inter_y1 = max(ax1,bx1), max(ay1,by1)
    inter_x2, inter_y2 = min(ax2,bx2), min(ay2,by2)
    inter = max(0, inter_x2-inter_x1) * max(0, inter_y2-inter_y1)
    area_a = max(0, ax2-ax1) * max(0, ay2-ay1)
    area_b = max(0, bx2-bx1) * max(0, by2-by1)
    union = area_a + area_b - inter + 1e-6
    return inter/union

def top1_hit(cands, gt, thr=0.5):
    if not cands: return 0
    b = cands[0]["xyxy"]
    return 1 if box_iou_xyxy(b, gt) >= thr else 0

## 8. DEMO: build pseudo manifests for the head

In [ ]:
import json, random

HEAD_TRAIN = f"{MANI_DIR}/head_train.json"
HEAD_VAL   = f"{MANI_DIR}/head_val.json"

if DEMO:
    # minimal pseudo data
    def center_gt(path):
        # pretend the center crop is GT
        return [160,120,480,360]
    train_entries=[]
    for p in train_imgs[:50]:
        train_entries.append({
            "image": p, "gt_box": center_gt(p),
            "verb_id": random.randint(0,4),
            "noun_id": 0,
            "ttc": round(random.uniform(0.2, 1.5), 2)
        })
    with open(HEAD_TRAIN,"w") as f: json.dump(train_entries, f, indent=2)

    with open(f"{MANI_DIR}/candidates_val_k5.json","r") as f:
        cand = json.load(f)
    val_entries=[]
    for e in cand[:30]:
        gt = e["candidates"][0]["xyxy"] if e["candidates"] else [0,0,1,1]
        val_entries.append({
            "image": e["image"],
            "gt_box": gt,
            "candidates": e["candidates"],
            "verb_id": random.randint(0,4),
            "noun_id": 0,
            "ttc": round(random.uniform(0.2, 1.5), 2)
        })
    with open(HEAD_VAL,"w") as f: json.dump(val_entries, f, indent=2)
    print("Wrote:", HEAD_TRAIN, HEAD_VAL)
else:
    print("DEMO=False: expecting real manifests at:", HEAD_TRAIN, HEAD_VAL)

## 9. Head model (ResNet18 crops ➜ MLP heads)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np, json
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

class SimpleHead(nn.Module):
    def __init__(self, hidden=256, num_verbs=5):
        super().__init__()
        rn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.trunk = nn.Sequential(*list(rn.children())[:-1])
        self.fc = nn.Linear(512, hidden)
        self.sc = nn.Linear(hidden, 1)
        self.vb = nn.Linear(hidden, num_verbs)
        self.tt = nn.Linear(hidden, 1)
    def forward(self, crops):
        x = self.trunk(crops)          # (N,512,1,1)
        x = x.view(x.size(0), -1)      # (N,512)
        x = F.relu(self.fc(x), inplace=True)
        score = self.sc(x).squeeze(-1)
        verb  = self.vb(x)
        ttc   = self.tt(x).squeeze(-1)
        return score, verb, ttc

preproc = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])

def crop_xyxy(img, box):
    x1,y1,x2,y2 = [int(v) for v in box]
    x1,y1 = max(0,x1), max(0,y1)
    x2,y2 = max(x1+1,x2), max(y1+1,y2)
    return img.crop((x1,y1,x2,y2))

def load_image(path):
    return Image.open(path).convert("RGB")

## 10. Head datasets

In [ ]:
class HeadTrainDS(torch.utils.data.Dataset):
    def __init__(self, manifest):
        self.items = json.load(open(manifest, "r"))
    def __len__(self): return len(self.items)
    def __getitem__(self, ix):
        it = self.items[ix]
        img = load_image(it["image"])
        crop = crop_xyxy(img, it["gt_box"])
        return {
            "crop": preproc(crop),
            "verb_id": torch.tensor(it["verb_id"]).long(),
            "ttc": torch.tensor(it["ttc"]).float(),
            "gt_box": torch.tensor(it["gt_box"]).float(),
        }

class HeadValDS(torch.utils.data.Dataset):
    def __init__(self, manifest):
        self.items = json.load(open(manifest, "r"))
    def __len__(self): return len(self.items)
    def __getitem__(self, ix):
        it = self.items[ix]
        img = load_image(it["image"])
        crops=[]
        for c in it.get("candidates", [])[:5]:
            crops.append(preproc(crop_xyxy(img, c["xyxy"])))
        if not crops:
            crops=[preproc(img)]
        crops = torch.stack(crops, dim=0)
        return {
            "crops": crops,
            "candidates": it.get("candidates", []),
            "gt_box": torch.tensor(it["gt_box"]).float(),
            "image": it["image"]
        }

## 11. Train head

In [ ]:
BATCH = 8 if DEMO else 16
EPOCHS_HEAD = 3 if DEMO else 20
LR=3e-4

train_ds = HeadTrainDS(HEAD_TRAIN)
val_ds   = HeadValDS(HEAD_VAL)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

head = SimpleHead().to(device)
opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=0.05)

def head_loss(score, verb, ttc, batch):
    ls = F.binary_cross_entropy_with_logits(score, torch.ones_like(score))
    lv = F.cross_entropy(verb, batch["verb_id"].to(verb.device))
    lt = F.smooth_l1_loss(ttc, batch["ttc"].to(ttc.device), beta=0.1)
    return ls+lv+lt, dict(l_score=ls.item(), l_verb=lv.item(), l_ttc=lt.item())

for ep in range(1, EPOCHS_HEAD+1):
    head.train()
    agg=[]
    for b in train_loader:
        x = b["crop"].to(device)
        s,v,t = head(x)
        loss, parts = head_loss(s,v,t,b)
        opt.zero_grad(); loss.backward(); opt.step()
        agg.append(loss.item())
    print(f"[{ep}/{EPOCHS_HEAD}] loss={np.mean(agg):.4f} parts={parts}")
# save
HEAD_CKPT = f"{CKPT_DIR}/trackA_head.pt"
torch.save(head.state_dict(), HEAD_CKPT)
print("Saved:", HEAD_CKPT)

## 12. Evaluate (Top1 IoU@0.5 proxy)

In [ ]:
head.eval()
hits=0; total=0
import numpy as np
with torch.no_grad():
    for b in val_loader:
        crops = b["crops"].to(device)
        s,v,t = head(crops)
        order = torch.argsort(s, descending=True).cpu().numpy().tolist()
        cand = [b["candidates"][0][i] for i in order] if b["candidates"] else []
        gt = b["gt_box"].squeeze(0).numpy().tolist()
        hits += top1_hit(cand, gt, 0.5)
        total += 1
print(f"Top1 IoU@0.5: {hits}/{total} = {hits/(total+1e-6):.3f}")

## 13. Track B placeholders (Fusion)

In [ ]:
import torch.nn as nn
class FusionHooks(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.lin = nn.Linear(dim, dim)
    def forward(self, img_tokens, vid_tokens):
        # TODO: implement FGTP and dual cross-attention
        return img_tokens, vid_tokens
print("Fusion hooks placeholder ready.")

## 14. Track C: RGTP-style pruning (demo)

In [ ]:
import torch
def prune_tokens(tokens, keep_ratio=0.5):
    B,N,D = tokens.shape
    keep = max(1, int(N*keep_ratio))
    norms = tokens.norm(dim=-1)  # (B,N)
    idx = torch.topk(norms, k=keep, dim=-1).indices
    out=[]
    for b in range(B):
        out.append(tokens[b, idx[b]])
    return torch.stack(out, dim=0), idx

B,N,D = 1, 128, 256
tok = torch.randn(B,N,D)
pruned, idx = prune_tokens(tok, 0.5)
print("Pruned:", tok.shape, "->", pruned.shape)